In [1]:
from google.colab import files
uploaded = files.upload()

Saving away_team.csv to away_team.csv
Saving away_team_score.csv to away_team_score.csv
Saving event.csv to event.csv
Saving home_team.csv to home_team.csv
Saving home_team_score.csv to home_team_score.csv
Saving odds.csv to odds.csv
Saving pbp.csv to pbp.csv
Saving power.csv to power.csv
Saving round.csv to round.csv
Saving season.csv to season.csv
Saving statistics.csv to statistics.csv
Saving time.csv to time.csv
Saving tournament.csv to tournament.csv
Saving venue.csv to venue.csv
Saving votes.csv to votes.csv


In [2]:
import pandas as pd

#uploaded file
files = {
    'event': 'event.csv',
    'home_team': 'home_team.csv',
    'away_team': 'away_team.csv',
    'home_team_score': 'home_team_score.csv',
    'away_team_score': 'away_team_score.csv',
    'time': 'time.csv',
    'tournament': 'tournament.csv',
    'venue': 'venue.csv',
    'votes': 'votes.csv',
    'round': 'round.csv',
    'season': 'season.csv',
    'odds': 'odds.csv',
    'power': 'power.csv',
}

data = {name: pd.read_csv(path) for name, path in files.items()}

#clean def
def dedupe_one_per_match(df, id_col='match_id'):
    # tekrari ha delete
    df = df.drop_duplicates()
    df = df.assign(_null_count=df.isnull().sum(axis=1))
    df = df.sort_values('_null_count').drop_duplicates(subset=id_col, keep='first')
    df = df.drop(columns='_null_count').reset_index(drop=True)
    return df

one_row_per_match = ['event', 'home_team', 'away_team', 'home_team_score',
                      'away_team_score', 'time', 'tournament', 'venue',
                      'votes', 'round', 'season']

for name in one_row_per_match:
    before = len(data[name])
    data[name] = dedupe_one_per_match(data[name])
    print(f"{name}: {before} -> {len(data[name])}")


for name in ['odds', 'power']:
    before = len(data[name])
    data[name] = data[name].drop_duplicates().reset_index(drop=True)
    print(f"{name}: {before} -> {len(data[name])}")

#normalys
data['round']['name'] = data['round']['name'].str.strip()
data['round']['name'] = data['round']['name'].replace({
    'Semifinals': 'Semifinal',
    'Quarterfinals': 'Quarterfinal',
})

data['event']['start_datetime'] = pd.to_datetime(data['event']['start_datetime'], unit='s')
data['home_team']['height'] = pd.to_numeric(data['home_team']['height'], errors='coerce')
data['away_team']['height'] = pd.to_numeric(data['away_team']['height'], errors='coerce')
data['home_team']['current_rank'] = pd.to_numeric(data['home_team']['current_rank'], errors='coerce')
data['away_team']['current_rank'] = pd.to_numeric(data['away_team']['current_rank'], errors='coerce')


for name, df in data.items():
    df.to_csv(f'{name}_clean.csv', index=False)

event: 35053 -> 16873
home_team: 25610 -> 12389
away_team: 24203 -> 11690
home_team_score: 35164 -> 16873
away_team_score: 35053 -> 16873
time: 35671 -> 16873
tournament: 35671 -> 16873
venue: 35423 -> 16749
votes: 35658 -> 16873
round: 19283 -> 9243
season: 35671 -> 16873
odds: 60946 -> 34807
power: 469677 -> 249587


**Question12**

In [3]:
import pandas as pd

period_cols = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']

#tafkik jensiat
gender_h = data['home_team'][['match_id', 'gender']].rename(columns={'gender': 'gender_h'})
gender_a = data['away_team'][['match_id', 'gender']].rename(columns={'gender': 'gender_a'})
gender = gender_h.merge(gender_a, on='match_id')

mismatch = gender[gender['gender_h'] != gender['gender_a']]
print(f"Matches with mismatched gender between home/away (likely mixed doubles or bad data): {len(mismatch)}")

gender_clean = gender[gender['gender_h'] == gender['gender_a']].dropna(subset=['gender_h'])
gender_clean = gender_clean.rename(columns={'gender_h': 'gender'})[['match_id', 'gender']]
print(f"Matches with consistent, valid gender: {len(gender_clean)}")
print(gender_clean['gender'].value_counts())

#long format brye har set
hs = data['home_team_score'][['match_id'] + period_cols]
as_ = data['away_team_score'][['match_id'] + period_cols]
merged = hs.merge(as_, on='match_id', suffixes=('_h', '_a'))

sets_long_list = []
for col in period_cols:
    sub = merged[['match_id', col + '_h', col + '_a']].dropna()
    sub = sub.rename(columns={col + '_h': 'home_games', col + '_a': 'away_games'})
    sub['games_in_set'] = sub['home_games'] + sub['away_games']
    sub['set_number'] = col
    sets_long_list.append(sub[['match_id', 'set_number', 'games_in_set']])

sets_long = pd.concat(sets_long_list, ignore_index=True)
print(f"\nTotal individual set observations: {len(sets_long)}")

#delete dade part
print("\nGames per set describe:")
print(sets_long['games_in_set'].describe())

#yek  set tennis 6ta20 game dre (ba hesab tybreak va set haye toolani bedoon tybreak)
outliers = sets_long[(sets_long['games_in_set'] < 6) | (sets_long['games_in_set'] > 30)]
print(f"Unrealistic set outliers (<6 or >30 games): {len(outliers)}")
sets_clean = sets_long[(sets_long['games_in_set'] >= 6) & (sets_long['games_in_set'] <= 30)]

#connect be jensiat
sets_with_gender = sets_clean.merge(gender_clean, on='match_id', how='inner')
print(f"\nSet observations after gender merge: {len(sets_with_gender)}")

# miangin ba tafkik jensiat
summary = sets_with_gender.groupby('gender')['games_in_set'].agg(['mean', 'count', 'std']).round(2)
print("\nGames per set by gender:")
print(summary)

men_avg = sets_with_gender[sets_with_gender['gender'] == 'M']['games_in_set'].mean()
women_avg = sets_with_gender[sets_with_gender['gender'] == 'F']['games_in_set'].mean()

#overall answer
print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Average games per set (Men)   : {men_avg:.2f}")
print(f"Average games per set (Women) : {women_avg:.2f}")
print(f"Difference (Men - Women)      : {men_avg - women_avg:.2f}")

Matches with mismatched gender between home/away (likely mixed doubles or bad data): 24
Matches with consistent, valid gender: 9858
gender
M    5545
F    4313
Name: count, dtype: int64

Total individual set observations: 36714

Games per set describe:
count    36714.000000
mean         9.477039
std          2.641404
min          0.000000
25%          8.000000
50%          9.000000
75%         10.000000
max         40.000000
Name: games_in_set, dtype: float64
Unrealistic set outliers (<6 or >30 games): 309

Set observations after gender merge: 21750

Games per set by gender:
        mean  count   std
gender                   
F       9.33   9529  2.21
M       9.71  12221  2.21

OVERALL ANSWER
Average games per set (Men)   : 9.71
Average games per set (Women) : 9.33
Difference (Men - Women)      : 0.38


**Question13**

In [4]:
import pandas as pd

# tarkib home&away + unique krdnshon
cols = ['player_id', 'name', 'plays', 'gender']
players = pd.concat([data['home_team'][cols], data['away_team'][cols]], ignore_index=True)
players_unique = players.drop_duplicates(subset='player_id', keep='first').reset_index(drop=True)
print(f"Unique players: {len(players_unique)}")

#faghat onaee ke data left hand va right hand drn bemonan
missing_count = players_unique['plays'].isnull().sum()
players_valid = players_unique.dropna(subset=['plays'])
players_valid = players_valid[players_valid['plays'].isin(['left-handed', 'right-handed'])]
print(f"Players with known handedness: {len(players_valid)} (excluded {missing_count} missing)")

#tozie
counts = players_valid['plays'].value_counts()
percentages = players_valid['plays'].value_counts(normalize=True) * 100
print("\nOverall distribution:")
print(counts)

#by gender(data ezafee)
by_gender = players_valid.groupby(['gender', 'plays']).size().unstack(fill_value=0)
by_gender_pct = players_valid.groupby('gender')['plays'].value_counts(normalize=True).unstack(fill_value=0) * 100
print("\nHandedness by gender (count):")
print(by_gender)
print("\nHandedness by gender (%):")
print(by_gender_pct.round(1))

#overall answer
right_count, left_count = counts.get('right-handed', 0), counts.get('left-handed', 0)
right_pct, left_pct = percentages.get('right-handed', 0), percentages.get('left-handed', 0)

print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Right-handed : {right_count} ({right_pct:.1f}%)")
print(f"Left-handed  : {left_count} ({left_pct:.1f}%)")
print("\nBy gender (%):")
print(by_gender_pct.round(1))

Unique players: 2644
Players with known handedness: 1146 (excluded 1498 missing)

Overall distribution:
plays
right-handed    1013
left-handed      133
Name: count, dtype: int64

Handedness by gender (count):
plays   left-handed  right-handed
gender                           
F                41           367
M                92           646

Handedness by gender (%):
plays   left-handed  right-handed
gender                           
F              10.0          90.0
M              12.5          87.5

OVERALL ANSWER
Right-handed : 1013 (88.4%)
Left-handed  : 133 (11.6%)

By gender (%):
plays   left-handed  right-handed
gender                           
F              10.0          90.0
M              12.5          87.5


**Question14**

In [5]:
import pandas as pd

#check krdn group turnoment
consistency_check = data['tournament'].groupby('tournament_id')['ground_type'].nunique()
print(f"Tournaments with inconsistent ground_type: {(consistency_check > 1).sum()}")

#faghat tornoment check kn na match
tournament_level = data['tournament'].drop_duplicates(subset='tournament_id').dropna(subset=['ground_type'])
print(f"Unique tournaments analyzed: {len(tournament_level)}")

#tozie
tour_counts = tournament_level['ground_type'].value_counts()
tour_pct = (tournament_level['ground_type'].value_counts(normalize=True) * 100).round(1)
print(pd.DataFrame({'count': tour_counts, 'pct': tour_pct}))

# overall answer
top_surface, top_count, top_pct = tour_counts.idxmax(), tour_counts.max(), tour_pct.max()
print("\n" + "="*75)
print("OVERALL ANSWER")
print("="*75)
print(f"Most common surface : {top_surface} — {top_count} tournaments ({top_pct}%)")

Tournaments with inconsistent ground_type: 6
Unique tournaments analyzed: 529
                   count   pct
ground_type                   
Hardcourt outdoor    239  45.2
Red clay             183  34.6
Hardcourt indoor      83  15.7
Grass                  8   1.5
Red clay indoor        6   1.1
Carpet indoor          5   0.9
Synthetic outdoor      4   0.8
Green clay             1   0.2

OVERALL ANSWER
Most common surface : Hardcourt outdoor — 239 tournaments (45.2%)
